# Movie Recommendation System

## Content-Based Recommendation using Machine Learning and NLP

This notebook demonstrates the complete implementation of a Content-Based Movie Recommendation System using the TMDB 5000 Movie Dataset.

Techniques used include:

- Data Preprocessing
- Feature Engineering
- Natural Language Processing (NLP)
- CountVectorizer
- Cosine Similarity
- Porter Stemming
- Model Serialization using Pickle

### 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import ast  ## Importing the "abstract syntax tree" (ast) module

### 2. Load Dataset

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv', on_bad_lines='skip', engine='python')
credits = pd.read_csv('tmdb_5000_credits.csv', on_bad_lines='skip', engine='python')

In [ ]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [ ]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 3. Merge Movie and Credits Dataset

In [ ]:
movies = movies.merge(credits,on ='title')

In [ ]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [ ]:
movies['original_language'].value_counts()

,count
original_language,
en,667
fr,2
ja,1
zh,1
es,1


In [ ]:
movies.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                672 non-null    int64  
 1   genres                672 non-null    object 
 2   homepage              385 non-null    object 
 3   id                    672 non-null    int64  
 4   keywords              672 non-null    object 
 5   original_language     672 non-null    object 
 6   original_title        672 non-null    object 
 7   overview              672 non-null    object 
 8   popularity            672 non-null    float64
 9   production_companies  672 non-null    object 
 10  production_countries  672 non-null    object 
 11  release_date          672 non-null    object 
 12  revenue               672 non-null    int64  
 13  runtime               672 non-null    float64
 14  spoken_languages      672 non-null    object 
 15  status                6

### 4. Data Cleaning and Feature Selection

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
movies.isnull().sum()

,0
movie_id,0
title,0
overview,0
genres,0
keywords,0
cast,0
crew,0


Now, i want to remove the NaN value from movies dataset. So, i will run this code

In [ ]:
movies.dropna(inplace=True) 

Now, the next command is for checking whether there are any duplicate data ?

In [ ]:
movies.duplicated().sum()

np.int64(0)

In [ ]:
movies.iloc[0].genres   # To select 1st row and only genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [ ]:
type(movies.iloc[0].genres)   

str

### 5. Feature Engineering

In [ ]:
def convert(obj):
  L = []
  for i in ast.literal_eval(obj):
    L.append( i['name'])
  return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

ValueError: malformed node or string: ['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']

In [ ]:
def convert_cast_list(obj):
  L = []
  counter = 0
  for i in ast.literal_eval(obj): # Convert string to Python list safely for 'cast'
    if counter != 3:
      L.append( i['name'])
      counter += 1
    else:
      break
  return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast_list)

In [ ]:
def fetch_director(obj):
  L = []
  for i in ast.literal_eval(obj): # Convert string to Python list for 'director'
    if i['job'] == 'Director':
      L.append( i['name'])
      break
  return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())  # Split each overview into a list of words using a short lambda function

In [ ]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [ ]:
# Define a function to remove spaces from all elements in a list
def remove_spaces(list_of_strings):
    return [i.replace(" ", "") for i in list_of_strings]

# Apply to all target columns
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(remove_spaces)

### 6. Create Tags Feature

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies_df = movies [['movie_id','title','tags']] # CREATE NEW DATAFRAME BCZ I WANT ONLY THIS THREE DATA

In [ ]:
movies_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [ ]:
# Join the list of words in 'tags' into a single string for each row
movies_df['tags'] = movies_df['tags'].apply(lambda x: " ".join(x))

/tmp/ipython-input-1060446550.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [ ]:
movies_df['tags'] = movies_df['tags'].apply(lambda x:x.lower())

/tmp/ipython-input-3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


### 7. Text Preprocessing using Porter Stemming

In [1]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()  # create stemmer object

def stem(text):
    y = []  # create empty list to hold stemmed words

    for i in text.split():  # split the text into words
        y.append(ps.stem(i))  # stem each word and add to list

    return " ".join(y)  # return the stemmed words as a single string

ModuleNotFoundError: No module named 'nltk'

In [ ]:
movies_df['tags'] = movies_df['tags'].apply(stem)

/tmp/ipython-input-3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [ ]:
movies_df['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [ ]:
movies_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


### 8. Text Vectorization

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors = cv.fit_transform(movies_df['tags']).toarray()

In [ ]:
# Display the words used as features (top 5000 frequent words excluding stop words)
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zookeeper', 'zorro', 'zoëkravitz'],
      dtype=object)

In [ ]:
# Import function to measure similarity between vectors using cosine angle
from sklearn.metrics.pairwise import cosine_similarity

### 9. Calculate Cosine Similarity

In [ ]:
# Calculate cosine similarity between all movie vectors
similarity = cosine_similarity(vectors)

# Show similarity scores of the first movie with all others
similarity[0]


array([1.        , 0.08035074, 0.08155909, 0.06584864, 0.16846773,
       0.10080973, 0.0355953 , 0.1287127 , 0.05356716, 0.09016696,
       0.09016696, 0.08662962, 0.08035074, 0.03874194, 0.12195122,
       0.05521576, 0.07144883, 0.12708382, 0.0895718 , 0.07362102,
       0.05396543, 0.10133892, 0.05521576, 0.08282364, 0.05001563,
       0.0473278 , 0.14126448, 0.16564729, 0.09756098, 0.059988  ,
       0.06497221, 0.1339179 , 0.07933635, 0.08700222, 0.        ,
       0.08414891, 0.15160183, 0.07502345, 0.07413766, 0.0722944 ,
       0.06834085, 0.1487679 , 0.        , 0.10726058, 0.03187884,
       0.079984  , 0.13199092, 0.18278756, 0.07933635, 0.05205792,
       0.10411584, 0.06697434, 0.13668171, 0.03874194, 0.0285133 ,
       0.06067575, 0.12022262, 0.02231054, 0.0631754 , 0.06584864,
       0.01664818, 0.21864327, 0.06512896, 0.07311503, 0.02804964,
       0.0176832 , 0.02145212, 0.14815944, 0.11043153, 0.02409813,
       0.07407972, 0.06984303, 0.12667365, 0.03062819, 0.19127

### 10. Recommendation Function

In [ ]:
def recommend(movie):
  movie = movie.strip()  # Remove extra spaces like 'Pirates '
  if movie not in movies_df['title'].values: # Check if the movie exists in the DataFrame
      print(f"❌ Movie '{movie}' not found in database.")
      return
  movie_index = movies_df[movies_df['title'] == movie].index[0]
  distances = similarity[movie_index]
  movies_list =  sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

  for i in movies_list:
    print(movies_df.iloc[i[0]].title)

In [ ]:
recommend('Avatar')

Titan A.E.
Independence Day
Battle: Los Angeles
Jupiter Ascending
The Fifth Element


### 11. Save Trained Objects

'Pickle' is a python library to save objects to disk.

Here we import pickle

In [ ]:
import pickle

Save movie dataframe

It saves processed movie data.

In [ ]:
pickle.dump(movies_df,open('movies.pkl','wb'))

Save Similarity Matrix

It Saves the brain of recommender.

So we don’t compute again every time.

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))